# Vision Inspection Portfolio - Training Notebook v3

## Instructions
- Run Cell 2 first (always)
- Run Cell 3 to mount Drive (always)
- Run Cell 4 only once to prepare data
- Run Cell 5, 6, 7 for training (can run independently)
- Run Cell 8 to export models
- Run Cell 9 to validate

In [ ]:
# Install dependencies - run this first always
!pip install -q ultralytics onnx

import ultralytics
import onnx
import torch
import cv2

print(f"Ultralytics version: {ultralytics.__version__}")
print(f"ONNX version: {onnx.__version__}")
print(f"PyTorch version: {torch.__version__}")
print(f"OpenCV version: {cv2.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

In [ ]:
# Mount Google Drive and verify folder structure
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Verify vision_portfolio folder exists
base_path = '/content/drive/MyDrive/vision_portfolio'
if os.path.exists(base_path):
    print(f"Found vision_portfolio at: {base_path}")
    print("\nDirectory contents:")
    for item in os.listdir(base_path):
        item_path = os.path.join(base_path, item)
        if os.path.isdir(item_path):
            print(f"  [DIR] {item}/")
        else:
            file_size = os.path.getsize(item_path) / (1024*1024)
            print(f"  [FILE] {item} ({file_size:.1f} MB)")
else:
    print(f"ERROR: vision_portfolio folder not found at: {base_path}")
    print("Please create the folder structure in Google Drive first")

In [ ]:
# Prepare datasets - run this once only
import os
import zipfile
import shutil
import cv2
import numpy as np
import yaml
import random
from sklearn.model_selection import train_test_split

# Set random seed for reproducibility
random.seed(42)
np.random.seed(42)

def mask_to_bbox(mask_path):
    """Convert segmentation mask to YOLO bounding box format"""
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return None
    
    # Threshold mask to binary
    _, binary_mask = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)
    
    # Find contours
    contours, _ = cv2.findContours(binary_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    if not contours:
        return None
    
    # Get bounding box of largest contour
    largest_contour = max(contours, key=cv2.contourArea)
    x, y, w, h = cv2.boundingRect(largest_contour)
    
    # Convert to YOLO format (normalized)
    img_h, img_w = mask.shape
    x_center = (x + w/2) / img_w
    y_center = (y + h/2) / img_h
    width = w / img_w
    height = h / img_h
    
    return x_center, y_center, width, height

# Step 1: Extract bottle.zip to /content/mvtec/bottle/
print("Step 1: Extracting bottle dataset...")
bottle_zip_path = '/content/drive/MyDrive/vision_portfolio/bottle.zip'
bottle_extract_path = '/content/mvtec/bottle'

if os.path.exists(bottle_extract_path):
    print("Bottle dataset already extracted, skipping")
else:
    if os.path.exists(bottle_zip_path):
        os.makedirs('/content/mvtec', exist_ok=True)
        with zipfile.ZipFile(bottle_zip_path, 'r') as zip_ref:
            zip_ref.extractall('/content/mvtec')
        print(f"Extracted bottle dataset to: {bottle_extract_path}")
    else:
        print(f"ERROR: bottle.zip not found at: {bottle_zip_path}")

# Step 2: Convert bottle to YOLO format
print("\nStep 2: Converting bottle to YOLO format...")
bottle_yolo_path = '/content/yolo/bottle'

if os.path.exists(bottle_yolo_path):
    print("Bottle YOLO dataset already exists, skipping")
else:
    bottle_classes = ['broken_large', 'broken_small', 'contamination']
    class_to_id = {cls: idx for idx, cls in enumerate(bottle_classes)}
    
    # Create YOLO directory structure
    for split in ['train', 'val']:
        for folder in ['images', 'labels']:
            os.makedirs(os.path.join(bottle_yolo_path, split, folder), exist_ok=True)
    
    train_count = 0
    val_count = 0
    
    # Process good images (all to train)
    good_train_path = os.path.join(bottle_extract_path, 'train', 'good')
    if os.path.exists(good_train_path):
        for img_file in os.listdir(good_train_path):
            if img_file.endswith('.png'):
                src_img = os.path.join(good_train_path, img_file)
                dst_img = os.path.join(bottle_yolo_path, 'train', 'images', img_file)
                shutil.copy2(src_img, dst_img)
                
                # Create empty label file
                label_file = img_file.replace('.png', '.txt')
                dst_label = os.path.join(bottle_yolo_path, 'train', 'labels', label_file)
                with open(dst_label, 'w') as f:
                    pass
                
                train_count += 1
    
    # Process defect images (80% train / 20% val)
    for defect_class in bottle_classes:
        test_defect_path = os.path.join(bottle_extract_path, 'test', defect_class)
        gt_defect_path = os.path.join(bottle_extract_path, 'ground_truth', defect_class)
        
        if os.path.exists(test_defect_path) and os.path.exists(gt_defect_path):
            defect_images = [f for f in os.listdir(test_defect_path) if f.endswith('.png')]
            
            # Split 80/20
            train_defects, val_defects = train_test_split(
                defect_images, test_size=0.2, random_state=42, shuffle=True
            )
            
            # Process train defects
            for img_file in train_defects:
                src_img = os.path.join(test_defect_path, img_file)
                dst_img = os.path.join(bottle_yolo_path, 'train', 'images', img_file)
                shutil.copy2(src_img, dst_img)
                
                mask_path = os.path.join(gt_defect_path, img_file)
                if os.path.exists(mask_path):
                    bbox = mask_to_bbox(mask_path)
                    if bbox:
                        label_file = img_file.replace('.png', '.txt')
                        dst_label = os.path.join(bottle_yolo_path, 'train', 'labels', label_file)
                        with open(dst_label, 'w') as f:
                            class_id = class_to_id[defect_class]
                            f.write(f"{class_id} {bbox[0]:.6f} {bbox[1]:.6f} {bbox[2]:.6f} {bbox[3]:.6f}\\n")
                
                train_count += 1
            
            # Process val defects
            for img_file in val_defects:
                src_img = os.path.join(test_defect_path, img_file)
                dst_img = os.path.join(bottle_yolo_path, 'val', 'images', img_file)
                shutil.copy2(src_img, dst_img)
                
                mask_path = os.path.join(gt_defect_path, img_file)
                if os.path.exists(mask_path):
                    bbox = mask_to_bbox(mask_path)
                    if bbox:
                        label_file = img_file.replace('.png', '.txt')
                        dst_label = os.path.join(bottle_yolo_path, 'val', 'labels', label_file)
                        with open(dst_label, 'w') as f:
                            class_id = class_to_id[defect_class]
                            f.write(f"{class_id} {bbox[0]:.6f} {bbox[1]:.6f} {bbox[2]:.6f} {bbox[3]:.6f}\\n")
                
                val_count += 1
    
    # Create dataset.yaml with local paths
    dataset_config = {
        'path': '/content/yolo/bottle',
        'train': '/content/yolo/bottle/train/images',
        'val': '/content/yolo/bottle/val/images',
        'nc': len(bottle_classes),
        'names': bottle_classes
    }
    
    with open(os.path.join(bottle_yolo_path, 'dataset.yaml'), 'w') as f:
        yaml.dump(dataset_config, f, default_flow_style=False)
    
    print(f"Bottle YOLO dataset created at: {bottle_yolo_path}")

# Step 3: Convert tile to YOLO format
print("\nStep 3: Converting tile to YOLO format...")
tile_raw_path = '/content/drive/MyDrive/vision_portfolio/mvtec/tile/tile'
tile_yolo_path = '/content/yolo/tile'

if os.path.exists(tile_yolo_path):
    print("Tile YOLO dataset already exists, skipping")
else:
    tile_classes = ['crack', 'glue_strip', 'gray_stroke', 'oil', 'rough']
    class_to_id = {cls: idx for idx, cls in enumerate(tile_classes)}
    
    # Create YOLO directory structure
    for split in ['train', 'val']:
        for folder in ['images', 'labels']:
            os.makedirs(os.path.join(tile_yolo_path, split, folder), exist_ok=True)
    
    train_count_tile = 0
    val_count_tile = 0
    
    # Process good images (all to train)
    good_train_path = os.path.join(tile_raw_path, 'train', 'good')
    if os.path.exists(good_train_path):
        for img_file in os.listdir(good_train_path):
            if img_file.endswith('.png'):
                src_img = os.path.join(good_train_path, img_file)
                dst_img = os.path.join(tile_yolo_path, 'train', 'images', img_file)
                shutil.copy2(src_img, dst_img)
                
                # Create empty label file
                label_file = img_file.replace('.png', '.txt')
                dst_label = os.path.join(tile_yolo_path, 'train', 'labels', label_file)
                with open(dst_label, 'w') as f:
                    pass
                
                train_count_tile += 1
    
    # Process defect images (80% train / 20% val)
    for defect_class in tile_classes:
        test_defect_path = os.path.join(tile_raw_path, 'test', defect_class)
        gt_defect_path = os.path.join(tile_raw_path, 'ground_truth', defect_class)
        
        if os.path.exists(test_defect_path) and os.path.exists(gt_defect_path):
            defect_images = [f for f in os.listdir(test_defect_path) if f.endswith('.png')]
            
            # Split 80/20
            train_defects, val_defects = train_test_split(
                defect_images, test_size=0.2, random_state=42, shuffle=True
            )
            
            # Process train defects
            for img_file in train_defects:
                src_img = os.path.join(test_defect_path, img_file)
                dst_img = os.path.join(tile_yolo_path, 'train', 'images', img_file)
                shutil.copy2(src_img, dst_img)
                
                mask_path = os.path.join(gt_defect_path, img_file)
                if os.path.exists(mask_path):
                    bbox = mask_to_bbox(mask_path)
                    if bbox:
                        label_file = img_file.replace('.png', '.txt')
                        dst_label = os.path.join(tile_yolo_path, 'train', 'labels', label_file)
                        with open(dst_label, 'w') as f:
                            class_id = class_to_id[defect_class]
                            f.write(f"{class_id} {bbox[0]:.6f} {bbox[1]:.6f} {bbox[2]:.6f} {bbox[3]:.6f}\\n")
                
                train_count_tile += 1
            
            # Process val defects
            for img_file in val_defects:
                src_img = os.path.join(test_defect_path, img_file)
                dst_img = os.path.join(tile_yolo_path, 'val', 'images', img_file)
                shutil.copy2(src_img, dst_img)
                
                mask_path = os.path.join(gt_defect_path, img_file)
                if os.path.exists(mask_path):
                    bbox = mask_to_bbox(mask_path)
                    if bbox:
                        label_file = img_file.replace('.png', '.txt')
                        dst_label = os.path.join(tile_yolo_path, 'val', 'labels', label_file)
                        with open(dst_label, 'w') as f:
                            class_id = class_to_id[defect_class]
                            f.write(f"{class_id} {bbox[0]:.6f} {bbox[1]:.6f} {bbox[2]:.6f} {bbox[3]:.6f}\\n")
                
                val_count_tile += 1
    
    # Create dataset.yaml with local paths
    dataset_config = {
        'path': '/content/yolo/tile',
        'train': '/content/yolo/tile/train/images',
        'val': '/content/yolo/tile/val/images',
        'nc': len(tile_classes),
        'names': tile_classes
    }
    
    with open(os.path.join(tile_yolo_path, 'dataset.yaml'), 'w') as f:
        yaml.dump(dataset_config, f, default_flow_style=False)
    
    print(f"Tile YOLO dataset created at: {tile_yolo_path}")

# Print final statistics
print("\nFinal dataset statistics:")
if os.path.exists('/content/yolo/bottle'):
    bottle_train_images = len([f for f in os.listdir('/content/yolo/bottle/train/images') if f.endswith('.png')])
    bottle_val_images = len([f for f in os.listdir('/content/yolo/bottle/val/images') if f.endswith('.png')])
    print(f"Bottle dataset - Train: {bottle_train_images} images, Val: {bottle_val_images} images")

if os.path.exists('/content/yolo/tile'):
    tile_train_images = len([f for f in os.listdir('/content/yolo/tile/train/images') if f.endswith('.png')])
    tile_val_images = len([f for f in os.listdir('/content/yolo/tile/val/images') if f.endswith('.png')])
    print(f"Tile dataset - Train: {tile_train_images} images, Val: {tile_val_images} images")

print("\nDataset preparation completed!")

In [ ]:
# Train bottle YOLOv8n model
from ultralytics import YOLO
import os
import shutil
import torch

# Verify GPU before starting
if not torch.cuda.is_available():
    print("WARNING: CUDA not available, training will be slow")
else:
    print(f"GPU available: {torch.cuda.get_device_name()}")

# Verify dataset exists
dataset_path = '/content/yolo/bottle/dataset.yaml'
if not os.path.exists(dataset_path):
    print(f"ERROR: Dataset not found at {dataset_path}")
    print("Please run Cell 4 first to prepare datasets")
else:
    print("Starting bottle YOLOv8n training...")
    
    # Initialize model
    model = YOLO('yolov8n.pt')
    
    # Train model - save to local storage for speed
    results = model.train(
        data=dataset_path,
        epochs=100,
        imgsz=640,
        batch=16,
        patience=20,
        device=0,
        project='/content/runs',
        name='bottle_n',
        exist_ok=True,
        hsv_h=0.015,
        hsv_s=0.7,
        hsv_v=0.4,
        flipud=0.3,
        fliplr=0.5,
        mosaic=1.0,
        mixup=0.1,
        save=True,
        plots=True
    )
    
    # Get final mAP50 score
    final_map50 = results.results_dict.get('metrics/mAP50(B)', 'N/A')
    print(f"\\nBottle YOLOv8n training completed")
    print(f"Final mAP50: {final_map50}")
    
    # Copy results to Google Drive
    local_path = '/content/runs/bottle_n'
    drive_path = '/content/drive/MyDrive/vision_portfolio/runs/bottle_n'
    
    if os.path.exists(local_path):
        os.makedirs('/content/drive/MyDrive/vision_portfolio/runs', exist_ok=True)
        
        # Remove existing drive folder if it exists
        if os.path.exists(drive_path):
            shutil.rmtree(drive_path)
        
        # Copy to drive
        shutil.copytree(local_path, drive_path)
        print(f"Results copied to: {drive_path}")
    else:
        print("WARNING: Local training results not found")

In [ ]:
# Train bottle YOLOv8s model
from ultralytics import YOLO
import os
import shutil
import torch

# Verify GPU before starting
if not torch.cuda.is_available():
    print("WARNING: CUDA not available, training will be slow")
else:
    print(f"GPU available: {torch.cuda.get_device_name()}")

# Verify dataset exists
dataset_path = '/content/yolo/bottle/dataset.yaml'
if not os.path.exists(dataset_path):
    print(f"ERROR: Dataset not found at {dataset_path}")
    print("Please run Cell 4 first to prepare datasets")
else:
    print("Starting bottle YOLOv8s training...")
    
    # Initialize model
    model = YOLO('yolov8s.pt')
    
    # Train model - save to local storage for speed
    results = model.train(
        data=dataset_path,
        epochs=100,
        imgsz=640,
        batch=16,
        patience=20,
        device=0,
        project='/content/runs',
        name='bottle_s',
        exist_ok=True,
        hsv_h=0.015,
        hsv_s=0.7,
        hsv_v=0.4,
        flipud=0.3,
        fliplr=0.5,
        mosaic=1.0,
        mixup=0.1,
        save=True,
        plots=True
    )
    
    # Get final mAP50 score
    final_map50 = results.results_dict.get('metrics/mAP50(B)', 'N/A')
    print(f"\\nBottle YOLOv8s training completed")
    print(f"Final mAP50: {final_map50}")
    
    # Copy results to Google Drive
    local_path = '/content/runs/bottle_s'
    drive_path = '/content/drive/MyDrive/vision_portfolio/runs/bottle_s'
    
    if os.path.exists(local_path):
        os.makedirs('/content/drive/MyDrive/vision_portfolio/runs', exist_ok=True)
        
        # Remove existing drive folder if it exists
        if os.path.exists(drive_path):
            shutil.rmtree(drive_path)
        
        # Copy to drive
        shutil.copytree(local_path, drive_path)
        print(f"Results copied to: {drive_path}")
    else:
        print("WARNING: Local training results not found")

In [ ]:
# Train tile YOLOv8n model
from ultralytics import YOLO
import os
import shutil
import torch

# Verify GPU before starting
if not torch.cuda.is_available():
    print("WARNING: CUDA not available, training will be slow")
else:
    print(f"GPU available: {torch.cuda.get_device_name()}")

# Verify dataset exists
dataset_path = '/content/yolo/tile/dataset.yaml'
if not os.path.exists(dataset_path):
    print(f"ERROR: Dataset not found at {dataset_path}")
    print("Please run Cell 4 first to prepare datasets")
else:
    print("Starting tile YOLOv8n training...")
    
    # Initialize model
    model = YOLO('yolov8n.pt')
    
    # Train model - save to local storage for speed
    results = model.train(
        data=dataset_path,
        epochs=100,
        imgsz=640,
        batch=16,
        patience=20,
        device=0,
        project='/content/runs',
        name='tile_n',
        exist_ok=True,
        hsv_h=0.015,
        hsv_s=0.7,
        hsv_v=0.4,
        flipud=0.3,
        fliplr=0.5,
        mosaic=1.0,
        mixup=0.1,
        save=True,
        plots=True
    )
    
    # Get final mAP50 score
    final_map50 = results.results_dict.get('metrics/mAP50(B)', 'N/A')
    print(f"\\nTile YOLOv8n training completed")
    print(f"Final mAP50: {final_map50}")
    
    # Copy results to Google Drive
    local_path = '/content/runs/tile_n'
    drive_path = '/content/drive/MyDrive/vision_portfolio/runs/tile_n'
    
    if os.path.exists(local_path):
        os.makedirs('/content/drive/MyDrive/vision_portfolio/runs', exist_ok=True)
        
        # Remove existing drive folder if it exists
        if os.path.exists(drive_path):
            shutil.rmtree(drive_path)
        
        # Copy to drive
        shutil.copytree(local_path, drive_path)
        print(f"Results copied to: {drive_path}")
    else:
        print("WARNING: Local training results not found")

In [ ]:
# Export all trained models to ONNX format with opset=21
from ultralytics import YOLO
import os
import onnx
import shutil

models_to_export = [
    {'name': 'bottle_n', 'model_path': '/content/runs/bottle_n/weights/best.pt'},
    {'name': 'bottle_s', 'model_path': '/content/runs/bottle_s/weights/best.pt'},
    {'name': 'tile_n', 'model_path': '/content/runs/tile_n/weights/best.pt'}
]

print("Exporting models to ONNX format with opset=21...\\n")

for model_info in models_to_export:
    model_name = model_info['name']
    model_path = model_info['model_path']
    
    if os.path.exists(model_path):
        print(f"Exporting {model_name}...")
        
        try:
            # Load model from local storage
            model = YOLO(model_path)
            
            # Export to ONNX with opset=21, simplify=True, imgsz=640
            local_onnx_path = model.export(format='onnx', opset=21, simplify=True, imgsz=640)
            
            # Verify opset version
            onnx_model = onnx.load(local_onnx_path)
            opset_version = onnx_model.opset_import[0].version
            
            # Get file size
            file_size = os.path.getsize(local_onnx_path) / (1024 * 1024)  # MB
            
            # Copy ONNX file to Google Drive
            drive_onnx_path = f'/content/drive/MyDrive/vision_portfolio/runs/{model_name}/weights/best.onnx'
            os.makedirs(os.path.dirname(drive_onnx_path), exist_ok=True)
            shutil.copy2(local_onnx_path, drive_onnx_path)
            
            print(f"  Local ONNX: {local_onnx_path}")
            print(f"  Drive ONNX: {drive_onnx_path}")
            print(f"  Opset version: {opset_version}")
            print(f"  File size: {file_size:.2f} MB\\n")
            
        except Exception as e:
            print(f"  ERROR exporting {model_name}: {str(e)}\\n")
    else:
        print(f"Skipping {model_name} - model file not found: {model_path}\\n")

print("Model export process completed")

In [ ]:
# Validate exported ONNX models
from ultralytics import YOLO
import os
import random

# Set random seed for reproducibility
random.seed(42)

models_to_validate = [
    {
        'name': 'bottle_n',
        'onnx_path': '/content/runs/bottle_n/weights/best.onnx',
        'test_categories': {
            'broken_large': '/content/mvtec/bottle/test/broken_large',
            'broken_small': '/content/mvtec/bottle/test/broken_small', 
            'contamination': '/content/mvtec/bottle/test/contamination'
        }
    },
    {
        'name': 'bottle_s',
        'onnx_path': '/content/runs/bottle_s/weights/best.onnx',
        'test_categories': {
            'broken_large': '/content/mvtec/bottle/test/broken_large',
            'broken_small': '/content/mvtec/bottle/test/broken_small',
            'contamination': '/content/mvtec/bottle/test/contamination'
        }
    },
    {
        'name': 'tile_n',
        'onnx_path': '/content/runs/tile_n/weights/best.onnx',
        'test_categories': {
            'crack': '/content/drive/MyDrive/vision_portfolio/mvtec/tile/tile/test/crack',
            'glue_strip': '/content/drive/MyDrive/vision_portfolio/mvtec/tile/tile/test/glue_strip',
            'gray_stroke': '/content/drive/MyDrive/vision_portfolio/mvtec/tile/tile/test/gray_stroke',
            'oil': '/content/drive/MyDrive/vision_portfolio/mvtec/tile/tile/test/oil',
            'rough': '/content/drive/MyDrive/vision_portfolio/mvtec/tile/tile/test/rough'
        }
    }
]

print("Validating exported ONNX models...\\n")

for model_info in models_to_validate:
    model_name = model_info['name']
    onnx_path = model_info['onnx_path']
    test_categories = model_info['test_categories']
    
    if os.path.exists(onnx_path):
        print(f"Validating {model_name}:")
        
        try:
            # Load ONNX model
            model = YOLO(onnx_path)
            
            # Test one random defect image per category
            for category, test_dir in test_categories.items():
                if os.path.exists(test_dir):
                    # Get random test image
                    test_images = [f for f in os.listdir(test_dir) if f.endswith('.png')]
                    if test_images:
                        test_image = random.choice(test_images)
                        test_image_path = os.path.join(test_dir, test_image)
                        
                        # Run inference with low confidence threshold
                        results = model(test_image_path, conf=0.01, verbose=False)
                        
                        # Analyze results
                        if results and len(results) > 0:
                            detections = results[0].boxes
                            if detections is not None and len(detections) > 0:
                                max_conf = float(detections.conf.max())
                                detection_count = len(detections)
                                
                                status = "PASS" if detection_count > 0 else "FAIL"
                                print(f"  {category}: {status} - {detection_count} detections, max conf: {max_conf:.3f}")
                            else:
                                print(f"  {category}: FAIL - No detections")
                        else:
                            print(f"  {category}: FAIL - No inference results")
                    else:
                        print(f"  {category}: SKIP - No test images found")
                else:
                    print(f"  {category}: SKIP - Test directory not found")
                    
        except Exception as e:
            print(f"  ERROR validating {model_name}: {str(e)}")
            
        print()  # Empty line for readability
    else:
        print(f"Skipping {model_name} - ONNX file not found: {onnx_path}\\n")

print("Model validation completed")